<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/04_split_by_bearing_py_(%2B_02_%EC%A0%84%EC%B2%B4_%EB%A3%A8%ED%94%84_%ED%86%B5%ED%95%A9).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# =============================================================
# 04_split_by_bearing.py  (+ 02 도메인 변환 전체 루프 통합)
# Paderborn(64kHz) -> IIS3DWB(26.6667kHz) 도메인 변환 + 누수방지 분할
# =============================================================
import os, glob, re, json
import numpy as np
from scipy.io import loadmat
from scipy.signal import resample_poly, butter, sosfiltfilt

# ============================================================
# [CONFIG] ★ 여기 3개만 확인하세요 ★
# ============================================================
MAT_DIR   = "/content/drive/My Drive/Colab Notebooks/Paderborn_raw_mat"   # 32개 베어링 폴더의 '루트'
OUT_DIR   = "/content/drive/My Drive/Colab Notebooks/Paderborn_iis3dwb"   # 변환 결과 저장 위치
DECLARED_UNIT = "g"          # 앞서 K001 분석으로 'g' 확정됨

# --- MVP 모드: True면 소량만 처리해 파이프라인 검증, False면 2560개 전체 ---
MVP_MODE = True
MVP_MAX_TRIALS_PER_BEARING = 2   # MVP일 때 베어링당 최대 파일 수

# --- 도메인 변환 파라미터 (02단계 확정값) ---
FS_IN     = 64000.0          # Paderborn 원본 샘플레이트
RESAMPLE_UP   = 5            # 5/12 = 26666.6667 Hz
RESAMPLE_DOWN = 12
FS_OUT    = FS_IN * RESAMPLE_UP / RESAMPLE_DOWN   # 26666.6667 Hz
LPF_CUTOFF = 6000.0          # IIS3DWB 대역 모사 (AAF 겸용)
NOISE_RMS_G = 0.0030         # 실측 정합 노이즈
SENSITIVITY_G_PER_LSB = 0.000488   # INT16 양자화 스텝
CLIP_G = 16.0                # IIS3DWB 풀스케일(±16g 가정)

# --- 분할 비율 & 시드 ---
SEED = 42
SPLIT_RATIO = {"train": 0.6, "val": 0.2, "test": 0.2}

rng_global = np.random.default_rng(SEED)


# ============================================================
# [1] 파일명 파서
#     예: N09_M07_F10_K001_1.mat
# ============================================================
def parse_filename(fname):
    stem = os.path.splitext(os.path.basename(fname))[0]
    parts = stem.split("_")
    if len(parts) < 5:
        return None
    speed, load, force, bearing, trial = parts[0], parts[1], parts[2], parts[3], parts[4]
    return {
        "speed": speed, "load": load, "force": force,
        "bearing": bearing, "trial": trial,
        "cond": f"{speed}_{load}_{force}",
        "label": 0 if bearing.startswith("K0") else 1,   # K0xx=정상, KA/KI=결함
    }


# ============================================================
# [2] 재귀 스캔:  root/{bearing}/{cond}/{file}.mat
# ============================================================
def scan_dataset(root_dir):
    if not os.path.isdir(root_dir):
        raise FileNotFoundError(f"경로 없음: {root_dir}\n → MAT_DIR을 확인하세요.")

    mat_files = sorted(glob.glob(os.path.join(root_dir, "**", "*.mat"), recursive=True))

    records = []
    for path in mat_files:
        meta = parse_filename(path)
        if meta is None:
            print(f"[SKIP] 파일명 형식 불일치: {os.path.basename(path)}")
            continue
        meta["path"] = path
        meta["fname"] = os.path.basename(path)
        records.append(meta)

    bearings = sorted(set(r["bearing"] for r in records))
    normal = [b for b in bearings if b.startswith("K0")]
    fault  = [b for b in bearings if not b.startswith("K0")]

    print("=" * 55)
    print(f"[SCAN] 총 파일: {len(records)}개")
    print(f"[SCAN] 베어링 종류: {len(bearings)}개")
    print(f"       정상({len(normal)}): {normal}")
    print(f"       결함({len(fault)}): {fault}")
    print(f"[SCAN] 운전조건: {sorted(set(r['cond'] for r in records))}")
    print("=" * 55)
    return records, normal, fault


# ============================================================
# [3] MAT 파일에서 vibration_1 채널 추출
#     Paderborn: 중첩 struct 구조
# ============================================================
def load_vibration(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    # 최상위 struct 키 찾기 (파일명과 동일한 변수명)
    keys = [k for k in mat.keys() if not k.startswith("__")]
    top = mat[keys[0]]
    # top.Y 는 신호 채널 배열, name 필드로 'vibration_1' 탐색
    Y = top.Y
    sig = None
    for ch in np.atleast_1d(Y):
        name = getattr(ch, "Name", "") or getattr(ch, "name", "")
        if str(name).lower() == "vibration_1":
            sig = np.asarray(ch.Data, dtype=np.float64).ravel()
            break
    if sig is None:
        raise ValueError(f"vibration_1 채널을 못 찾음: {path}")
    return sig


# ============================================================
# [4] 도메인 변환 (02단계 로직)
#     6kHz LPF -> resample_poly -> gain -> noise -> clip -> INT16
# ============================================================
_sos = butter(8, LPF_CUTOFF, btype="low", fs=FS_IN, output="sos")

def convert_file_to_iis3dwb(sig_g, randomize=False, seed=None):
    rng = np.random.default_rng(seed) if seed is not None else rng_global

    x = sig_g.astype(np.float64)
    x = sosfiltfilt(_sos, x)                              # 1) 6kHz LPF (AAF)
    x = resample_poly(x, RESAMPLE_UP, RESAMPLE_DOWN)      # 2) 64k -> 26.6667k
    if randomize:                                         # 3) gain(증강, Train만)
        x = x * rng.uniform(0.95, 1.05)
    x = x + rng.normal(0.0, NOISE_RMS_G, size=x.shape)    # 4) 노이즈
    x = np.clip(x, -CLIP_G, CLIP_G)                       # 5) 클리핑
    q = np.round(x / SENSITIVITY_G_PER_LSB).astype(np.int16)  # 6) INT16 양자화
    return q


# ============================================================
# [5] 누수 방지 분할: 베어링 단위로 Train/Val/Test 분리
#     각 split에 정상+결함이 모두 포함되도록 보장
# ============================================================
def split_by_bearing(normal, fault):
    assert len(normal) >= 3, f"정상 베어링 부족(최소 3): {normal}"
    assert len(fault)  >= 3, f"결함 베어링 부족(최소 3): {fault}"

    def _split(bearing_list):
        b = list(bearing_list)
        rng_global.shuffle(b)
        n = len(b)
        n_tr = max(1, int(round(n * SPLIT_RATIO["train"])))
        n_va = max(1, int(round(n * SPLIT_RATIO["val"])))
        # 나머지는 test (최소 1개 확보)
        n_te = n - n_tr - n_va
        if n_te < 1:
            n_te = 1
            n_tr = n - n_va - n_te
        return b[:n_tr], b[n_tr:n_tr+n_va], b[n_tr+n_va:]

    n_tr, n_va, n_te = _split(normal)
    f_tr, f_va, f_te = _split(fault)

    split = {
        "train": set(n_tr + f_tr),
        "val":   set(n_va + f_va),
        "test":  set(n_te + f_te),
    }

    # --- 누수 검증: 겹치는 베어링 없어야 함 ---
    all_b = list(split["train"]) + list(split["val"]) + list(split["test"])
    assert len(all_b) == len(set(all_b)), "누수 발생! 베어링이 여러 split에 중복됨"

    # --- 각 split에 정상/결함 모두 포함되는지 검증 ---
    for name, s in split.items():
        has_normal = any(b.startswith("K0") for b in s)
        has_fault  = any(not b.startswith("K0") for b in s)
        assert has_normal, f"[{name}] split에 정상 베어링 없음: {sorted(s)}"
        assert has_fault,  f"[{name}] split에 결함 베어링 없음: {sorted(s)}"

    print("[SPLIT] 베어링 분할 결과")
    for name, s in split.items():
        print(f"   {name:5s}: {sorted(s)}")
    return split


# ============================================================
# [6] 전체 루프: 스캔 -> 분할 -> 변환 -> 저장
# ============================================================
def run_pipeline():
    records, normal, fault = scan_dataset(MAT_DIR)
    if len(records) == 0:
        raise RuntimeError("스캔된 파일이 0개입니다. MAT_DIR / 드라이브 마운트 확인 필요.")

    split = split_by_bearing(normal, fault)

    # 각 레코드에 split 태그 부여
    def which_split(bearing):
        for name, s in split.items():
            if bearing in s:
                return name
        return None

    # MVP 모드: 베어링당 파일 수 제한
    if MVP_MODE:
        by_bearing = {}
        kept = []
        for r in records:
            by_bearing.setdefault(r["bearing"], 0)
            if by_bearing[r["bearing"]] < MVP_MAX_TRIALS_PER_BEARING:
                by_bearing[r["bearing"]] += 1
                kept.append(r)
        records = kept
        print(f"[MVP] 베어링당 최대 {MVP_MAX_TRIALS_PER_BEARING}개 제한 → 처리 대상 {len(records)}개")

    os.makedirs(OUT_DIR, exist_ok=True)
    manifest = []
    n_ok, n_fail = 0, 0

    for i, r in enumerate(records):
        sp = which_split(r["bearing"])
        if sp is None:
            continue
        try:
            sig = load_vibration(r["path"])
            # Train만 증강(randomize), 파일별 고유 시드
            is_train = (sp == "train")
            file_seed = SEED + abs(hash(r["fname"])) % 100000
            q = convert_file_to_iis3dwb(sig, randomize=is_train, seed=file_seed)

            out_name = f"{sp}__{r['bearing']}__{r['cond']}__{r['trial']}.npy"
            out_path = os.path.join(OUT_DIR, out_name)
            np.save(out_path, q)

            manifest.append({
                "split": sp, "bearing": r["bearing"], "label": r["label"],
                "cond": r["cond"], "trial": r["trial"],
                "n_samples": int(q.shape[0]), "out": out_name,
            })
            n_ok += 1
            if (i + 1) % 20 == 0:
                print(f"   ...진행 {i+1}/{len(records)}")
        except Exception as e:
            n_fail += 1
            print(f"[FAIL] {r['fname']}: {e}")

    # manifest 저장
    with open(os.path.join(OUT_DIR, "manifest.json"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print("=" * 55)
    print(f"[DONE] 변환 성공 {n_ok}개 / 실패 {n_fail}개")
    print(f"[DONE] fs_out = {FS_OUT:.4f} Hz, unit = {DECLARED_UNIT}")
    print(f"[DONE] 저장 위치: {OUT_DIR}")
    print(f"[DONE] manifest.json 생성 완료")
    print("=" * 55)
    return manifest


# ============================================================
# 실행
# ============================================================
if __name__ == "__main__":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    run_pipeline()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[SCAN] 총 파일: 2561개
[SCAN] 베어링 종류: 32개
       정상(6): ['K001', 'K002', 'K003', 'K004', 'K005', 'K006']
       결함(26): ['KA01', 'KA03', 'KA04', 'KA05', 'KA06', 'KA07', 'KA08', 'KA09', 'KA15', 'KA16', 'KA22', 'KA30', 'KB23', 'KB24', 'KB27', 'KI01', 'KI03', 'KI04', 'KI05', 'KI07', 'KI08', 'KI14', 'KI16', 'KI17', 'KI18', 'KI21']
[SCAN] 운전조건: ['N09_M07_F10', 'N15_M01_F10', 'N15_M07_F04', 'N15_M07_F10']
[SPLIT] 베어링 분할 결과
   train: ['K003', 'K004', 'K005', 'K006', 'KA03', 'KA04', 'KA06', 'KA07', 'KA08', 'KA09', 'KA15', 'KA16', 'KA22', 'KI03', 'KI05', 'KI07', 'KI08', 'KI16', 'KI17', 'KI18']
   val  : ['K002', 'KA01', 'KA05', 'KI01', 'KI04', 'KI21']
   test : ['K001', 'KA30', 'KB23', 'KB24', 'KB27', 'KI14']
[MVP] 베어링당 최대 2개 제한 → 처리 대상 64개
   ...진행 20/64
   ...진행 40/64
   ...진행 60/64
[DONE] 변환 성공 64개 / 실패 0개
[DONE] fs_out = 26666.6667 Hz, unit = g
[DONE] 저장 위치: /content/